# 🧠 Module 5: Memory & Context Management

---

## 🎯 Objective
Enable AI to remember and use past conversation context.


## ⚙️ Setup (Reuse LLM)

In [1]:
import requests

def ask_llm(prompt, model="mistral"):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": model,
            "prompt": prompt,
            "stream": False
        }
    )
    return response.json()["response"]

/Users/abhishekjha/Documents/ai-learning/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


#### 🧪 Step 1: Basic Chat Memory
---
👉 Store conversation as a list

In [40]:
# Store conversation history
chat_history = []

def add_to_history(user, ai):
    """
    Store user and AI messages
    """
    chat_history.append({
        "user": user,
        "ai": ai
    })

#### 🧪 Step 2: Inject Memory into Prompt
---
👉 Pass history to LLM

In [67]:
def build_prompt(user_input):
    """
    Create prompt with conversation history
    """
    
    history_text = ""
    
    # Convert history into text format
    for chat in chat_history:
        history_text += f"User: {chat['user']}\n"
        history_text += f"AI: {chat['ai']}\n"
    
    # Add current query
    prompt = f"""
    You are a helpful assistant. 
    You respond exactly what is asked and avoid unnecessary commentary. 
    Avoid making guess if you dont know.
    
    Conversation so far:
    {history_text}

    User: {user_input}
    AI:
    """
    
    return prompt

In [68]:
# Chat function with memory
def chat(user_input):
    """
    Chat function with memory
    """
    
    prompt = build_prompt(user_input)
    
    response = ask_llm(prompt)
    
    # Save conversation
    add_to_history(user_input, response)
    
    return response

In [69]:
# Reset chat history to avoid unnecessary commentry
chat_history = []
# Test memory
print(chat("My name is Abhishek"), "\n")
print(chat("What is my name?"))  # should remember

 Hello Abhishek, it's nice to meet you! How can I assist you today? 

 Your name is Abhishek.


**⚠️ Problem: Memory Keeps Growing**

- 👉 More history = slower + expensive
- 👉 Models have token limits

#### 🧪 Step 3: Sliding Window Memory
---
👉 Keep only last N conversations

In [70]:
MAX_HISTORY = 3

def add_to_history_limited(user, ai):
    chat_history.append({"user": user, "ai": ai})
    
    # Keep only last N items
    while len(chat_history) > MAX_HISTORY:
        chat_history.pop(0)

In [71]:
# Chat function with sliding window memory
def chat_with_window(user_input):
    """
    Chat function with memory
    """
    
    prompt = build_prompt(user_input)
    
    response = ask_llm(prompt)
    
    # Save conversation
    add_to_history_limited(user_input, response)
    
    return response

### Senario 1: Hard Memory Loss
---
💡 Expected Behavior
- AI likely says:
    - “I don’t know”
    - Or guesses

In [72]:
# Reset chat history to avoid unnecessary commentry
chat_history = []

# Senario 1: Hard Memory Loss
print("AI:", chat_with_window("My name is Abhishek"), "\n")
print("AI:", chat_with_window("I live in Bangalore"), "\n")
print("AI:", chat_with_window("I work as a developer"), "\n")

# These will push out older memory
print("AI:", chat_with_window("I like cricket"), "\n")
print("AI:", chat_with_window("I enjoy traveling"), "\n")

# Now ask
print("AI:", chat_with_window("What is my name?"), "\n")

# Debug: see what memory contains
print("\nCurrent Memory:")
print(chat_history)

AI:  Hello Abhishek, it's nice to meet you! How can I assist you today? 

AI:  You currently reside in Bangalore. Is there anything specific you need help with regarding this location or would you like to discuss something else? 

AI:  Great! As a developer, if you have any questions related to coding, programming languages, software development, or need suggestions for tools or resources, feel free to ask. If your question is not directly related to that, please let me know what you'd like to discuss instead. 

AI:  Cricket is a popular sport in India and many people enjoy watching or playing it. If you have any questions about cricket matches, teams, players, rules, or if you would like to discuss anything related to cricket, feel free to ask! If your question is not directly related to cricket, please let me know what else you'd like to talk about instead. 

AI:  Traveling can be a great way to explore new places, experience different cultures, and broaden our horizons. If you have 

### Scenario 2: Partial Context Loss
---
)
💡 Expected Behavior
- AI may:
    - Answer location correctly
    - Miss the name

👉 Because:
- Name was pushed out
- Location still exists

In [74]:
print("\n--- Sliding Window: Partial Context Loss ---\n")

chat_history = []

print("AI:", chat_with_window("My name is Abhishek"), "\n")
print("AI:", chat_with_window("I live in Bangalore"), "\n")
print("AI:", chat_with_window("I work in AI"), "\n")

# Push only first entry out
print("AI:", chat_with_window("I like Python"), "\n")

# Ask mixed question
print("AI:", chat_with_window("Where do I live and what is my name?"), "\n")


--- Sliding Window: Partial Context Loss ---

AI:  Hello Abhishek, it's nice to meet you! How can I assist you today? 

AI:  You currently reside in Bangalore. 

AI:  You are employed in the field of Artificial Intelligence. 

AI:  You prefer the programming language Python. 

AI:  Your current residence is Bangalore, but I don't have information about your name. 



### Scenario 3: Conflicting Context (Window Bias)
---

💡 Expected Behavior
- AI will likely say: "Rahul"

👉 Because:
- Sliding window prioritizes latest info only

In [75]:
# Reset chat history to avoid unnecessary commentry
chat_history = []

# Senario 2
print("AI:", chat_with_window("My name is Abhishek"), "\n")
print("AI:", chat_with_window("My name is Rahul"), "\n")
print("AI:", chat_with_window("I work as a developer"), "\n")

print("AI:", chat_with_window("What is my name?"), "\n")

AI:  Hello Abhishek, it's nice to meet you! 

AI:  Hello Rahul, it's nice to meet you too! 

AI:  Hello Rahul, it's good to know that you work as a developer. 

AI:  Your name is Rahul, as per our previous conversation. 



#### 🧪 Step 4: Summarized Memory
---
👉 Instead of deleting, summarize old chats

In [59]:
summary_memory = ""

def update_summary():
    """
    Summarize older chats
    """
    
    global summary_memory
    
    prompt = f"""
    Summarize the following conversation briefly:

    {chat_history}
    """
    
    summary_memory = ask_llm(prompt)

In [63]:
# chat with summary + Window
def chat_with_summary(user_input):
    """
    Chat with sliding window + summary memory
    """
    
    global summary_memory
    
    # Build prompt with summary + recent chats
    history_text = ""
    for chat in chat_history:
        history_text += f"User: {chat['user']}\nAI: {chat['ai']}\n"
    
    prompt = f"""
    You are a helpful assistant.

    Summary of previous conversation:
    {summary_memory}

    Recent conversation:
    {history_text}

    User: {user_input}
    AI:
    """
    
    response = ask_llm(prompt)
    
    # Add to history
    add_to_history_limited(user_input, response)
    
    # Update summary after every 3 interactions
    if len(chat_history) == MAX_HISTORY:
        update_summary()
    
    return response

### Scenario 1: Sliding Window Forgetting
---
👉 Add more than 3 interactions → older info disappears

💡 Expected Behavior
- AI may fail to recall your name
Because:
- It got pushed out of window
- Summary may not include it

In [76]:
# Reset chat history to avoid unnecessary commentry
chat_history = []
summary_memory = ""

print("AI:", chat_with_summary("My name is Abhishek"), "\n")
print("AI:", chat_with_summary("I live in Bangalore"), "\n")
print("AI:", chat_with_summary("I work as a software developer"), "\n")

# These push older memory out
print("AI:", chat_with_summary("I like cricket"), "\n")
print("AI:", chat_with_summary("I enjoy traveling"), "\n")

# Now ask old info
print("AI:", chat_with_summary("What is my name?"), "\n")

print("\nSummary Memory:")
print(summary_memory)


--- Scenario 1: Sliding Window Forgetting ---

AI:  Hello Abhishek! It's great to meet you. I am here to help you with your questions and provide assistance on various topics, especially those related to AI and software development. Let's dive into what interests you the most. Would you like some guidance on a specific project or concept you're currently working on, or perhaps recommendations for online courses to boost your AI skills? I can also help explain complex concepts, offer suggestions for design patterns, algorithms, data structures, or even discuss current trends in AI and Machine Learning. Don't hesitate to ask if there's something specific you have in mind! 

AI:  Hello Abhishek! It seems we share the same city, Bangalore. That's fantastic! With a vibrant tech scene and numerous opportunities for growth in AI, there are plenty of resources available to us. Let me provide some recommendations that could be useful for your AI journey:

1. Local AI Meetups/Workshops: Attend 

### Scenario 3: Ambiguous Query (Context Confusion)
---
💡 Expected Behavior
AI may respond:
- Abhishek
- Rahul
- Confused answer

👉 Because query lacks clarity

In [77]:
chat_history = []
summary_memory = ""

print("AI:", chat_with_summary("My name is Abhishek"), "\n")
print("AI:", chat_with_summary("My friend's name is Rahul"), "\n")
print("AI:", chat_with_summary("We both work in tech"), "\n")

print("AI:", chat_with_summary("What is the name?"), "\n")

AI:  Hello Abhishek, it's nice to meet you! How can I assist you today? If you have any questions or need help with something, feel free to ask. 

AI:  Hello Rahul, it's great to make your acquaintance as well! If Abhishek needs assistance or has any questions, he can certainly ask me. But if you have something you'd like help with, feel free to ask too! I'm here to help. 

AI:  That's interesting! Working in technology can offer a wide range of opportunities and challenges. If you or Abhishek have any specific questions related to your work, or need advice on certain topics within the field, I'm more than happy to help! Here are a few examples:

- Programming languages you might be interested in learning
- Best practices for project management
- Tips for career growth and networking
- Resources for staying updated on industry trends
- Solutions to common problems faced in your roles.

Just let me know what you're looking for, and I'll do my best to assist! 

AI:  I apologize for any c

## ⚠️ Real-World Challenges

- Memory may confuse model
- Old info may be lost
- Summaries may degrade accuracy